# SOLiD-PINN v0.9.0a — direct-download recovery notebook
CPU. Run all. No Google Drive mount, no GitHub PAT, no private data. Same pinned solver, 72 fits, four tests and mesh refinement as Notebook 10. Failed numerical mesh gate is a recorded outcome, not an export failure. JSON is stored in the temporary Colab runtime and downloaded directly; download before ending the session.


In [ ]:
from google.colab import files
from pathlib import Path
import os,sys,subprocess,tempfile,json,hashlib
REV="1b73989fc730e02954e1187e9409e0da01fa31ff"
repo=Path(tempfile.mkdtemp(prefix="solid-v090-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
hashes={"hybrid_p2d.py":"20aaeede052a29a2b65b1cd322b9fade3b4094fd4e1564d15787d7e240c51f18","test_hybrid_p2d.py":"08f898a7771aea81ebb41d92d375fe758b1dc8d7663a45321096062d5be4295f"}
for k,v in hashes.items(): assert hashlib.sha256((repo/"scripts"/k).read_bytes()).hexdigest()==v
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_hybrid_p2d.py"],cwd=repo,env=env,check=True)
out=repo/"results.json"
subprocess.run([sys.executable,"scripts/hybrid_p2d.py","--out",str(out)],cwd=repo,env=env,check=True)
d=json.loads(out.read_text())
assert len(d["metrics"])==72 and all(x["success"] for x in d["metrics"])
assert {(x["truth"],x["noise"],x["seed"],x["method"]) for x in d["metrics"]}=={(t,n,s,m) for t in ("null","rest","peak") for n in (1,5) for s in (11,23,37) for m in ("calibrated","linear","rbf","oracle")}
# Independent axial/radial and joint refinement. Report failure, never hide it.
mesh_script=repo/"scripts"/"_colab_mesh.py"
mesh_script.write_text("""import json,numpy as np
from hybrid_p2d import cell,profile,BLOCKS
out={}
for k in BLOCKS:
    j=profile(k)
    v={str((n,m)):cell(j,n,m)[0] for n,m in [(4,6),(8,6),(4,12),(8,12),(16,12),(8,24),(16,24)]}
    pairs=[((4,6),(8,6)),((4,6),(4,12)),((4,6),(8,12)),((8,12),(16,12)),((8,12),(8,24)),((8,12),(16,24))]
    out[k]={str(a)+'->'+str(b):float(np.max(abs(v[str(a)]-v[str(b)]))*1000) for a,b in pairs}
print(json.dumps(out))
""")
mesh=json.loads(subprocess.check_output([sys.executable,str(mesh_script)],cwd=repo,env=env,text=True))
d.update(source_revision=REV,source_hashes=hashes,targeted_tests_passed=4,mesh=mesh,mesh_gate_passed=all(max(list(x.values())[-3:])<=1 for x in mesh.values()))
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/SOLiD-PINN/v090/handoff")
root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v090_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    tmp=dest.with_suffix(".part");tmp.write_bytes(raw);tmp.replace(dest)
assert dest.read_bytes()==raw
print("MESH GATE",d["mesh_gate_passed"],"LOCAL FILE VERIFIED",dest.name,sha)
try:
    files.download(str(dest))
except Exception as e:
    print("Automatic download unavailable:", str(e))
    print("Download manually from the Colab Files panel:", str(dest))
